# SPOT Evaluation - Adversarial Edits (T = 1)

Adversarial edit budgets are evaluated on the saved recomputed pivots and the single `GT` label.

This notebook evaluates **SPOT-plugin**, using the Li et al. optimal-weight fraction estimator, and **SPOT-oracle**. It sweeps the original calibration grid, selects operating points using the FPR-bin rule at target FPR 0.05, and times each selected operating point.

**Input:** `post_edit_adversarial_T1.zip`  
**Output:** `evaluation_adversarial_SPOT_T1.zip`


In [ ]:
%pip -q install pandas


In [ ]:
import json
import shutil
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


In [ ]:
def locate_zip(filename: str) -> Path:
    candidates = [Path('/content') / filename, Path('/mnt/data') / filename, Path.cwd() / filename]
    for path in candidates:
        if path.exists() and zipfile.is_zipfile(path):
            return path
    try:
        from google.colab import files
        uploaded = files.upload()
        for uploaded_name in uploaded:
            path = Path('/content') / uploaded_name
            if path.suffix.lower() == '.zip' and zipfile.is_zipfile(path):
                return path
    except Exception:
        pass
    raise FileNotFoundError(f'Could not find {filename}.')


def load_dataset_zip(filename: str, workdir_name: str):
    zip_path = locate_zip(filename)
    root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    workdir = root / workdir_name
    if workdir.exists():
        shutil.rmtree(workdir)
    workdir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(workdir)

    dataset_paths = list(workdir.rglob('dataset.npz'))
    meta_paths = list(workdir.rglob('meta.json'))
    if len(dataset_paths) != 1 or len(meta_paths) != 1:
        raise FileNotFoundError('The input ZIP must contain one dataset.npz and one meta.json.')
    return (
        np.load(dataset_paths[0], allow_pickle=False),
        json.loads(meta_paths[0].read_text(encoding='utf-8')),
        zip_path.stem,
    )


def zip_output_folder(folder: Path, zip_name: str) -> Path:
    root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
    zip_path = root / zip_name
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix('')), 'zip', root_dir=folder)
    print('Saved:', zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
    return zip_path


In [ ]:
def histogram_density(data: np.ndarray, bins: int = 500):
    values = np.clip(np.asarray(data, dtype=float).reshape(-1), 0.0, 1.0)
    counts, edges = np.histogram(values, bins=bins, range=(0.0, 1.0), density=False)
    width = edges[1] - edges[0]
    density = counts / (values.size * width + 1e-12)
    return density, edges


def evaluate_histogram_density(values: np.ndarray, edges: np.ndarray,
                               density: np.ndarray) -> np.ndarray:
    indices = np.searchsorted(edges, np.asarray(values).reshape(-1), side='right') - 1
    indices = np.clip(indices, 0, density.size - 1)
    return density[indices]


# Optimal-weight fraction estimator of Li et al.
def prepare_fraction_estimator(reference_pivots: np.ndarray, bins: int = 500,
                                      uniform_size: int = 200_000,
                                      uniform_seed: int = 2024):
    reference = np.asarray(reference_pivots, dtype=float).reshape(-1)
    density, edges = histogram_density(reference, bins)
    uniform = np.random.default_rng(uniform_seed).random(uniform_size)
    return {
        'density': density,
        'edges': edges,
        'uniform_density': evaluate_histogram_density(uniform, edges, density),
        'reference_density': evaluate_histogram_density(reference, edges, density),
    }


def estimate_fraction(observed_pivots: np.ndarray, prepared,
                             max_iterations: int = 200,
                             tolerance: float = 1e-8) -> float:
    observed_density = evaluate_histogram_density(
        np.asarray(observed_pivots, dtype=float).reshape(-1),
        prepared['edges'],
        prepared['density'],
    )
    uniform_density = prepared['uniform_density']
    reference_density = prepared['reference_density']

    def update(epsilon: float) -> float:
        E0 = np.mean((1.0 - uniform_density) / ((1.0 - epsilon) + epsilon * uniform_density))
        EP = np.mean((1.0 - reference_density) / ((1.0 - epsilon) + epsilon * reference_density))
        EY = np.mean((1.0 - observed_density) / ((1.0 - epsilon) + epsilon * observed_density))
        next_epsilon = (E0 - EY) / (E0 - EP)
        return float(np.clip(next_epsilon, 1e-3, 1.0))

    epsilon = 0.5
    for _ in range(max_iterations):
        next_epsilon = update(epsilon)
        if abs(next_epsilon - epsilon) < tolerance:
            epsilon = next_epsilon
            break
        epsilon = next_epsilon
    return float(epsilon)


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class SPOTParameters:
    u_min: float = 0.05
    u_max: float = 0.95
    grid_multiplier: float = 20.0
    grid_power: float = 1.0
    eta_multiplier: float = 1.0


SPOT_PARAMETERS = SPOTParameters()


def prepare_spot_base(pivots: np.ndarray, parameters: SPOTParameters = SPOT_PARAMETERS):
    pivots = np.asarray(pivots, dtype=float).reshape(-1)
    n = int(pivots.size)
    log_n = max(1.0, float(np.log(n)))
    grid_size = int(np.ceil(parameters.grid_multiplier * (log_n ** parameters.grid_power)))
    grid_size = max(10, min(5000, grid_size))
    u_grid = np.linspace(parameters.u_min, parameters.u_max, grid_size)
    thresholds = np.clip(1.0 - n ** (-u_grid), 0.0, 1.0)

    sorted_pivots = np.sort(pivots)
    indices = np.searchsorted(sorted_pivots, thresholds, side='right')
    empirical_tail = (n - indices) / n
    base = (1.0 - thresholds) / np.maximum(empirical_tail, 1.0 / n)
    return {'n': n, 'log_n': log_n, 'thresholds': thresholds, 'base': base}


def spot_decisions_from_base(pivots: np.ndarray, prepared, epsilon_hat: float,
                             calibration_constant: float,
                             parameters: SPOTParameters = SPOT_PARAMETERS) -> np.ndarray:
    lambda_n = float(calibration_constant) / prepared['log_n']
    eta_n = parameters.eta_multiplier / prepared['log_n'] ** 2
    estimated_fdr = (1.0 - float(np.clip(epsilon_hat, 0.0, 1.0))) * prepared['base']
    admissible = np.flatnonzero(estimated_fdr <= lambda_n - eta_n)
    index = int(admissible[0]) if admissible.size else len(prepared['thresholds']) - 1
    return np.asarray(pivots) > float(prepared['thresholds'][index])


def run_spot(pivots: np.ndarray, epsilon_hat: float, calibration_constant: float) -> np.ndarray:
    prepared = prepare_spot_base(pivots)
    return spot_decisions_from_base(pivots, prepared, epsilon_hat, calibration_constant)


In [ ]:
def token_metrics(prediction: np.ndarray, ground_truth: np.ndarray) -> dict:
    prediction = np.asarray(prediction, dtype=bool)
    ground_truth = np.asarray(ground_truth, dtype=bool)
    tp = int(np.sum(prediction & ground_truth))
    fp = int(np.sum(prediction & ~ground_truth))
    tn = int(np.sum(~prediction & ~ground_truth))
    fn = int(np.sum(~prediction & ground_truth))
    return {
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'true_count': tp + fn,
        'IoU': float(tp / (tp + fp + fn + 1e-12)),
        'FPR': float(fp / (fp + tn + 1e-12)),
    }


def summarize_metrics(metrics: pd.DataFrame, parameter_column: str) -> pd.DataFrame:
    group_columns = ['method', 'case', 'level_index', 'edit_level', parameter_column]
    grouped = metrics.groupby(group_columns, dropna=False, as_index=False)
    summary = grouped.agg(
        TP_mean=('TP', 'mean'),
        TP_sum=('TP', 'sum'),
        true_sum=('true_count', 'sum'),
        IoU=('IoU', 'mean'),
        FPR=('FPR', 'mean'),
        n_documents=('document_index', 'count'),
    )
    summary['TPR'] = summary['TP_sum'] / summary['true_sum'].replace(0, np.nan)
    return summary


def select_best_with_fpr_bin(summary: pd.DataFrame, metric: str,
                              parameter_column: str,
                              target_fpr: float = 0.05,
                              bin_width: float = 0.01) -> pd.DataFrame:
    bin_uppers = np.round(np.arange(bin_width, target_fpr + 1e-12, bin_width), 2)
    records = []
    group_columns = ['method', 'case', 'level_index', 'edit_level']

    for group_values, group in summary.groupby(group_columns, dropna=False, sort=True):
        chosen = None
        used_upper = np.nan
        fallback_steps = np.nan
        for index in range(len(bin_uppers) - 1, -1, -1):
            upper = float(bin_uppers[index])
            lower = round(upper - bin_width, 2)
            if lower <= 0:
                mask = (group['FPR'] >= lower - 1e-12) & (group['FPR'] <= upper + 1e-12)
            else:
                mask = (group['FPR'] > lower) & (group['FPR'] <= upper + 1e-12)
            candidates = group[mask]
            if not candidates.empty:
                candidates = candidates.sort_values(
                    [metric, 'FPR', parameter_column],
                    ascending=[False, False, True],
                )
                chosen = candidates.iloc[0]
                used_upper = upper
                fallback_steps = len(bin_uppers) - 1 - index
                break

        record = dict(zip(group_columns, group_values))
        record.update({
            'selection_metric': metric,
            'target_FPR': target_fpr,
            'used_bin_lower': used_upper - bin_width if np.isfinite(used_upper) else np.nan,
            'used_bin_upper': used_upper,
            'fallback_steps': fallback_steps,
            parameter_column: np.nan,
            'selected_FPR': np.nan,
            'selected_IoU': np.nan,
            'selected_TPR': np.nan,
        })
        if chosen is not None:
            record.update({
                parameter_column: float(chosen[parameter_column]),
                'selected_FPR': float(chosen['FPR']),
                'selected_IoU': float(chosen['IoU']),
                'selected_TPR': float(chosen['TPR']),
            })
        records.append(record)

    return pd.DataFrame(records)


## Configuration


In [ ]:
INPUT_ZIP_NAME = 'post_edit_adversarial_T1.zip'
OUTPUT_ZIP_NAME = 'evaluation_adversarial_SPOT_T1.zip'

C_FINE = np.round(np.arange(0.01, 1.00 + 1e-12, 0.01), 2)
C_COARSE = np.round(np.arange(1.00, 5.50 + 1e-12, 0.05), 2)
C_VALUES = sorted(set(float(value) for value in np.concatenate([C_FINE, C_COARSE])))
TARGET_FPR = 0.05
REFERENCE_BINS = 500
MAX_REFERENCE_PIVOTS = 50_000
REFERENCE_SEED = 123


In [ ]:
data, meta, input_stem = load_dataset_zip(INPUT_ZIP_NAME, 'post_edit_adversarial_T1_extracted')
if not {'Ys', 'S1'}.issubset(data.files):
    raise KeyError('The input must contain Ys and S1.')
if float(meta.get('temp', 1.0)) != 1.0:
    raise ValueError('This notebook is configured for T = 1.')

edit_budgets = [int(value) for value in meta.get('top_k_list', [5, 10, 15, 20, 30, 40])]
root = Path('/content') if Path('/content').exists() else Path('/mnt/data')
output_folder = root / 'evaluation_adversarial_SPOT_T1'
if output_folder.exists():
    shutil.rmtree(output_folder)
output_folder.mkdir(parents=True, exist_ok=True)

samples = []
for level_index, budget in enumerate(edit_budgets):
    pivot_key = f'Ys_post_adv_repo_k{budget}'
    gt_key = f'GT_adv_repo_k{budget}'
    if pivot_key not in data.files or gt_key not in data.files:
        raise KeyError(f'Missing {pivot_key} or {gt_key}.')
    pivots = np.asarray(data[pivot_key])
    ground_truth = np.asarray(data[gt_key]).astype(bool)
    for document_index in range(pivots.shape[0]):
        Y = np.asarray(pivots[document_index], dtype=float).reshape(-1)
        GT = np.asarray(ground_truth[document_index], dtype=bool).reshape(-1)
        keep = np.isfinite(Y)
        samples.append({
            'case': 'adversarial_edits',
            'level_index': int(level_index),
            'edit_level': float(budget),
            'document_index': int(document_index),
            'Y': Y[keep],
            'GT': GT[keep],
        })
print('Samples:', len(samples))


In [ ]:
reference_pivots = np.asarray(data['Ys'])[np.asarray(data['S1']).astype(bool)].reshape(-1)
reference_pivots = reference_pivots[np.isfinite(reference_pivots)]
reference_pivots = np.clip(reference_pivots, 0.0, 1.0)
if reference_pivots.size == 0:
    raise RuntimeError('The pre-edit dataset contains no watermarked reference pivots.')
if reference_pivots.size > MAX_REFERENCE_PIVOTS:
    reference_pivots = np.random.default_rng(REFERENCE_SEED).choice(
        reference_pivots, MAX_REFERENCE_PIVOTS, replace=False
    )

fraction_reference = prepare_fraction_estimator(
    reference_pivots, bins=REFERENCE_BINS
)
print('Reference pivots:', reference_pivots.size)


## Evaluate, select operating points, and measure runtime


In [ ]:
# Precompute the fraction estimates and the C-independent SPOT arrays.
for sample in samples:
    sample['epsilon_true'] = float(np.clip(np.mean(sample['GT']), 0.0, 1.0))
    sample['epsilon_plugin'] = estimate_fraction(
        sample['Y'], fraction_reference
    )
    sample['spot_base'] = prepare_spot_base(sample['Y'])

fraction_rows = [
    {
        'case': sample['case'],
        'level_index': sample['level_index'],
        'edit_level': sample['edit_level'],
        'document_index': sample['document_index'],
        'epsilon_true': sample['epsilon_true'],
        'epsilon_plugin': sample['epsilon_plugin'],
    }
    for sample in samples
]
fraction_per_sample = pd.DataFrame(fraction_rows)
fraction_per_sample.to_csv(output_folder / 'fraction_per_sample.csv', index=False)

fraction_summary_rows = []
for group_values, group in fraction_per_sample.groupby(
    ['case', 'level_index', 'edit_level'], dropna=False, sort=True
):
    record = dict(zip(['case', 'level_index', 'edit_level'], group_values))
    true_mean = float(group['epsilon_true'].mean())
    record.update({
        'epsilon_true_mean': true_mean,
        'epsilon_plugin_mean': float(group['epsilon_plugin'].mean()),
        'relative_RMSE_plugin': float(
            np.sqrt(np.mean((group['epsilon_plugin'] - group['epsilon_true']) ** 2))
            / (true_mean + 1e-12)
        ),
        'n_documents': int(len(group)),
    })
    fraction_summary_rows.append(record)
fraction_summary = pd.DataFrame(fraction_summary_rows)
fraction_summary.to_csv(output_folder / 'fraction_summary.csv', index=False)

method_epsilon_key = {
    'SPOT-plugin': 'epsilon_plugin',
    'SPOT-oracle': 'epsilon_true',
}

all_summary = []
for calibration_constant in C_VALUES:
    rows = []
    for sample in samples:
        for method, epsilon_key in method_epsilon_key.items():
            prediction = spot_decisions_from_base(
                sample['Y'],
                sample['spot_base'],
                sample[epsilon_key],
                calibration_constant,
            )
            rows.append({
                'method': method,
                'case': sample['case'],
                'level_index': sample['level_index'],
                'edit_level': sample['edit_level'],
                'document_index': sample['document_index'],
                'C': float(calibration_constant),
                'epsilon_hat': float(sample[epsilon_key]),
                **token_metrics(prediction, sample['GT']),
            })

    metrics = pd.DataFrame(rows)
    summary = summarize_metrics(metrics, 'C')
    all_summary.append(summary)

    c_tag = str(calibration_constant).replace('.', 'p')
    c_folder = output_folder / f'C{c_tag}'
    c_folder.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(c_folder / 'metrics.csv', index=False)
    summary.to_csv(c_folder / 'metrics_summary.csv', index=False)

summary_all = pd.concat(all_summary, ignore_index=True)
summary_all.to_csv(output_folder / 'metrics_summary_all_C.csv', index=False)

selected = pd.concat([
    select_best_with_fpr_bin(summary_all, 'IoU', 'C', TARGET_FPR),
    select_best_with_fpr_bin(summary_all, 'TPR', 'C', TARGET_FPR),
], ignore_index=True)
selected.to_csv(output_folder / 'selected_parameters.csv', index=False)
display(selected)

# Time each selected operating point. File loading and the parameter sweep are excluded.
samples_by_group = {}
for sample in samples:
    key = (sample['case'], sample['level_index'])
    samples_by_group.setdefault(key, []).append(sample)

runtime_rows = []
for row in selected.itertuples(index=False):
    calibration_constant = float(row.C)
    if not np.isfinite(calibration_constant):
        continue
    group_samples = samples_by_group[(row.case, row.level_index)]

    def run_one(sample):
        if row.method == 'SPOT-plugin':
            epsilon_hat = estimate_fraction(sample['Y'], fraction_reference)
        else:
            epsilon_hat = float(np.mean(sample['GT']))
        return run_spot(sample['Y'], epsilon_hat, calibration_constant)

    run_one(group_samples[0])  # untimed warm-up
    started = time.perf_counter()
    for sample in group_samples:
        run_one(sample)
    elapsed = time.perf_counter() - started

    runtime_rows.append({
        'method': row.method,
        'case': row.case,
        'level_index': row.level_index,
        'edit_level': row.edit_level,
        'selection_metric': row.selection_metric,
        'C': calibration_constant,
        'runtime_seconds_total': elapsed,
        'runtime_seconds_per_document': elapsed / len(group_samples),
        'n_documents': len(group_samples),
    })

runtime_columns = [
    'method', 'case', 'level_index', 'edit_level', 'selection_metric',
    'C', 'runtime_seconds_total', 'runtime_seconds_per_document', 'n_documents',
]
runtime_by_level = pd.DataFrame(runtime_rows, columns=runtime_columns)
runtime_by_level.to_csv(output_folder / 'runtime_by_level.csv', index=False)
display(runtime_by_level)

metadata = {
    'input_dataset': INPUT_ZIP_NAME,
    'temperature': 1.0,
    'methods': list(method_epsilon_key),
    'fraction_estimator': 'Li et al. optimal-weight estimator',
    'C_values': C_VALUES,
    'target_FPR': TARGET_FPR,
    'selection_rule': (
        'Use the highest nonempty 0.01-wide FPR bin not exceeding the target, '
        'then maximize IoU or TPR within that bin.'
    ),
}
(output_folder / 'summary.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)
zip_output_folder(output_folder, OUTPUT_ZIP_NAME)
